In [2]:
import numpy as np
import os

def load_graph(filepath):
    """Načte graf ze souboru, kde každý řádek je odkaz ve tvaru 'odkud,kam' (1-based indexy)."""
    edges = []
    max_node = 0
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            if len(parts) != 2:
                continue
            u, v = int(parts[0]), int(parts[1])
            edges.append((u, v))
            max_node = max(max_node, u, v)
    
    # Uzly jsou očíslované od 1, převedeme na 0-based
    N = max_node
    A = np.zeros((N, N), dtype=float)
    for u, v in edges:
        A[u - 1, v - 1] = 1.0
    return A

def compute_pagerank(A, alpha=0.2, max_iter=100, tol=1e-10):
    """Vypočítá PageRank vektorizovaně v Numpy."""
    N = A.shape[0]
    out_degrees = A.sum(axis=1, keepdims=True)
    
    # Stránka bez odkazů má alpha = 1 (chová se jako teleportér na všechny stránky)
    no_out = (out_degrees == 0).flatten()
    
    # Matice přechodu M (řádky vydělené počtem odkazů)
    M = np.zeros_like(A)
    has_out = ~no_out
    if np.any(has_out):
        M[has_out] = A[has_out] / out_degrees[has_out]
    
    # Stránky bez odkazů rozdělí váhu rovnoměrně všude
    if np.any(no_out):
        M[no_out] = 1.0 / N
        
    # A' = (1 - alpha) * M + (alpha / N) * E
    E = np.ones((N, N), dtype=float) / N
    A_prime_T = ((1.0 - alpha) * M + alpha * E).T
    
    # Počáteční vektor p(0) = [1/N, 1/N, ..., 1/N]T
    p = np.ones(N, dtype=float) / N
    
    converged_at = max_iter
    for i in range(max_iter):
        p_next = A_prime_T @ p
        diff = np.sum(np.abs(p_next - p))
        if diff < tol:
            converged_at = i + 1
            break
        p = p_next
        
    return p, converged_at

def compute_hits(A, max_iter=100):
    """Vypočítá HITS (huby a autority) vektorizovaně v Numpy."""
    N = A.shape[0]
    a = np.ones(N, dtype=float)
    h = np.ones(N, dtype=float)
    
    for _ in range(max_iter):
        h_next = A @ a
        a_next = A.T @ h
        
        # Normalizace součtem obou vektorů
        h = h_next / np.sum(h_next)
        a = a_next / np.sum(a_next)
        
    return a, h

# --- HLAVNÍ SKRIPT ---
if __name__ == "__main__":
    data_dir = "./data"
    
    for graph_id in range(1, 7):
        filepath = os.path.join(data_dir, f"graph_{graph_id}.txt")
        if not os.path.exists(filepath):
            print(f"Soubor {filepath} neexistuje, přeskakuji.")
            continue
            
        A = load_graph(filepath)
        N = A.shape[0]
        
        # Výpočet PageRanku
        pr, conv_iter = compute_pagerank(A, alpha=0.2, max_iter=100)
        
        # Výpočet HITS
        authorities, hubs = compute_hits(A, max_iter=100)
        
        print(f"=== GRAF {graph_id} (Počet uzlů: {N}) ===")
        print(f"PageRank se přestal měnit po iteraci: {conv_iter}")
        print(f"PageRank hodnoty: {np.round(pr, 4)}")
        print(f"Autority (A):     {np.round(authorities, 4)}")
        print(f"Huby (H):         {np.round(hubs, 4)}")
        
        # Pro graf 6 vypíšeme top 5 uzlů
        if graph_id == 6:
            top_pr = np.argsort(pr)[::-1][:5] + 1  # zpět na 1-based indexy
            top_auth = np.argsort(authorities)[::-1][:5] + 1
            top_hubs = np.argsort(hubs)[::-1][:5] + 1
            in_degrees = np.sum(A, axis=0)
            
            print("\n--- DETAIL PRO GRAF 6 (Top 5) ---")
            print(f"Top 5 PageRank uzlů: {top_pr} (hodnoty: {np.round(pr[top_pr-1], 4)})")
            print(f"Top 5 Autority:      {top_auth} (hodnoty: {np.round(authorities[top_auth-1], 4)})")
            print(f"Top 5 Huby:          {top_hubs} (hodnoty: {np.round(hubs[top_hubs-1], 4)})")
            print(f"Počet příchozích odkazů pro tyto uzly: {[int(in_degrees[node-1]) for node in top_pr]}")
        print("-" * 50)

=== GRAF 1 (Počet uzlů: 6) ===
PageRank se přestal měnit po iteraci: 43
PageRank hodnoty: [0.0656 0.1181 0.1601 0.1937 0.2205 0.242 ]
Autority (A):     [0.  0.2 0.2 0.2 0.2 0.2]
Huby (H):         [0.2 0.2 0.2 0.2 0.2 0. ]
--------------------------------------------------
=== GRAF 2 (Počet uzlů: 5) ===
PageRank se přestal měnit po iteraci: 1
PageRank hodnoty: [0.2 0.2 0.2 0.2 0.2]
Autority (A):     [0.2 0.2 0.2 0.2 0.2]
Huby (H):         [0.2 0.2 0.2 0.2 0.2]
--------------------------------------------------
=== GRAF 3 (Počet uzlů: 4) ===
PageRank se přestal měnit po iteraci: 26
PageRank hodnoty: [0.1786 0.3214 0.3214 0.1786]
Autority (A):     [0.191 0.309 0.309 0.191]
Huby (H):         [0.191 0.309 0.309 0.191]
--------------------------------------------------
=== GRAF 4 (Počet uzlů: 7) ===
PageRank se přestal měnit po iteraci: 27
PageRank hodnoty: [0.2726 0.1567 0.1384 0.1092 0.1853 0.0656 0.0722]
Autority (A):     [0.1395 0.1779 0.2008 0.1402 0.2014 0.0561 0.0841]
Huby (H):       

=== GRAF 6 (Počet uzlů: 1228) ===
PageRank se přestal měnit po iteraci: 22
PageRank hodnoty: [0.0007 0.0007 0.0007 ... 0.0007 0.002  0.0007]
Autority (A):     [0.     0.0009 0.     ... 0.0002 0.0137 0.    ]
Huby (H):         [0.0027 0.     0.     ... 0.     0.0097 0.    ]

--- DETAIL PRO GRAF 6 (Top 5) ---
Top 5 PageRank uzlů: [1052  761 1151   62   78] (hodnoty: [0.0036 0.003  0.003  0.0029 0.0029])
Top 5 Autority:      [ 761 1151   62   78  394] (hodnoty: [0.0304 0.0304 0.0302 0.03   0.0293])
Top 5 Huby:          [171 857 185  91  79] (hodnoty: [0.0162 0.0155 0.0154 0.0153 0.0152])
Počet příchozích odkazů pro tyto uzly: [89, 68, 68, 68, 67]
--------------------------------------------------
